In [1]:
# ==============================================================================
# 1. 라이브러리 설치 및 데이터 다운로드
# ==============================================================================

!pip install efficientnet_pytorch
!pip install torchmetrics
!pip install kagglehub  # KaggleHub 설치

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import kagglehub # 임포트
import time
import copy
from google.colab import drive
from collections import Counter # ⚠️ 클래스 카운트를 위한 라이브러리 추가
from google.colab import drive #

# 다운로드 및 경로 설정
path = kagglehub.dataset_download("sujaykapadnis/cucumber-disease-recognition-dataset")

print("다운로드된 데이터셋 경로:", path)

# GPU 사용 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=cb6deea78dbdd9f99d3d605c1d9767be916129af961a9a62ae2d1a7fc08c12dd
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet_pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 57.8 MB/s eta 0:00:00
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


100%|██████████| 2.59G/2.59G [02:04<00:00, 22.3MB/s]

Extracting files...


다운로드된 데이터셋 경로: /root/.cache/kagglehub/datasets/sujaykapadnis/cucumber-disease-recognition-dataset/versions/1
사용 장치: cuda:0


In [20]:
# ==============================================================================
# 2. 데이터셋 경로 설정 (✨ 오이 모델 최종 확정: 중첩 폴더 반영)
# ==============================================================================
import os

# 'path'는 1번 섹션에서 다운로드한 루트 경로입니다.
DATA_ROOT_DOWNLOAD = path
print(f"DEBUG: 다운로드 루트 경로: {DATA_ROOT_DOWNLOAD}")

# 1. 중간 폴더 경로
INTERMEDIATE_FOLDER = 'Cucumber Disease Recognition Dataset'
INTERMEDIATE_PATH = os.path.join(DATA_ROOT_DOWNLOAD, INTERMEDIATE_FOLDER)

# 2. 🚨 [수정] 중첩된 폴더('Augmented Image') 경로를 포함합니다.
AUGMENTED_FOLDER_1 = 'Augmented Image'
AUGMENTED_FOLDER_2 = 'Augmented Image' # 👈 중첩된 폴더

# ImageFolder가 스캔할 최종 경로
DATA_ROOT_FOR_SPLIT = os.path.join(INTERMEDIATE_PATH, AUGMENTED_FOLDER_1, AUGMENTED_FOLDER_2)

print(f"\n✅ 최종 데이터 로드 경로 (DATA_ROOT_FOR_SPLIT): {DATA_ROOT_FOR_SPLIT}")


# 3. 최종 경로 검증
if not os.path.exists(DATA_ROOT_FOR_SPLIT):
    print(f"❌ 오류: DATA_ROOT_FOR_SPLIT 경로를 찾을 수 없습니다: {DATA_ROOT_FOR_SPLIT}")
else:
    print(f"✅ 경로 확인 완료: {DATA_ROOT_FOR_SPLIT}")
    print("     '4. 데이터 로드' 섹션을 실행하세요.")

DEBUG: 다운로드 루트 경로: /root/.cache/kagglehub/datasets/sujaykapadnis/cucumber-disease-recognition-dataset/versions/1

✅ 최종 데이터 로드 경로 (DATA_ROOT_FOR_SPLIT): /root/.cache/kagglehub/datasets/sujaykapadnis/cucumber-disease-recognition-dataset/versions/1/Cucumber Disease Recognition Dataset/Augmented Image/Augmented Image
✅ 경로 확인 완료: /root/.cache/kagglehub/datasets/sujaykapadnis/cucumber-disease-recognition-dataset/versions/1/Cucumber Disease Recognition Dataset/Augmented Image/Augmented Image
     '4. 데이터 로드' 섹션을 실행하세요.


In [21]:
# ==============================================================================
# 3. 하이퍼파라미터 및 데이터 변환 설정 (모델 축소 적용)
# ==============================================================================
from efficientnet_pytorch import EfficientNet

# 하이퍼파라미터
BATCH_SIZE = 32 # B0 모델에 맞춰 배치 사이즈를 32로 늘려 속도 개선
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001 # 과적합 방지를 위해 학습률은 계속 낮게 유지
MODEL_NAME = 'efficientnet-b0' # ⭐️ 모델을 EfficientNet-B0로 축소

# EfficientNet-B0의 권장 입력 이미지 크기 (224x224로 자동 변경됨)
INPUT_SIZE = EfficientNet.get_image_size(MODEL_NAME)

# 데이터 증강 및 정규화
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.RandomResizedCrop(INPUT_SIZE),
        # 💡 [추가] 회전 및 색상 변화 추가
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ImageNet 통계
    ]),
    'test': transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


In [22]:
# ==============================================================================
# 4. 데이터 로드 및 DataLoader 생성 (수정: 단일 폴더 구조 → Random Split)
# ==============================================================================
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

print("\n데이터 로딩 및 안전한 90:10 분할 중...")

# 1. DATA_ROOT 전체를 하나의 데이터셋으로 로드합니다.
# ⚠️ [수정] DATA_ROOT 대신, 클래스 폴더가 들어있는 정확한 경로 변수를 사용해야 합니다.
#    이전 경로 설정 섹션에서 정의한 DATA_ROOT_FOR_SPLIT 변수를 사용합니다.
full_dataset = ImageFolder(DATA_ROOT_FOR_SPLIT, data_transforms['train']) # 💡 DATA_ROOT_FOR_SPLIT 사용
full_size = len(full_dataset)

# 2. 비율 설정 (90% 학습, 10% 검증)
train_size = int(0.9 * full_size)
val_size = full_size - train_size

# 3. 데이터셋 분리 (겹치지 않음을 보장)
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

# 4. DataLoader 재정의
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True),
    # 'val' 키에 검증 데이터셋(val_dataset)을 로드합니다.
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
}

# 5. 메타 정보 업데이트
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)

print(f"클래스 개수: {NUM_CLASSES}")
print(f"클래스 이름: {class_names}")
print(f"✅ 새로운 학습 데이터 크기: {dataset_sizes['train']} (90%)")
print(f"✅ 새로운 검증 데이터 크기: {dataset_sizes['val']} (10%)")


데이터 로딩 및 안전한 90:10 분할 중...
클래스 개수: 8
클래스 이름: ['Anthracnose', 'Bacterial Wilt', 'Belly Rot', 'Downy Mildew', 'Fresh Cucumber', 'Fresh Leaf', 'Gummy Stem Blight', 'Pythium Fruit Rot']
✅ 새로운 학습 데이터 크기: 5760 (90%)
✅ 새로운 검증 데이터 크기: 640 (10%)


In [23]:
# ==============================================================================
# 5. 모델 설정 및 가중치 부여 손실 함수 설정 (✨ B0 모델로 축소)
# ==============================================================================
from collections import Counter
import torch.nn as nn
from efficientnet_pytorch import EfficientNet

# ==============================================================================
# 1. 클래스 가중치 계산 (Class Imbalance 대처)
# ==============================================================================
from collections import Counter
# ⚠️ image_datasets['train'] 대신 full_dataset을 사용합니다.
# full_dataset은 '4. 데이터 로드' 섹션에서 정의되어야 합니다.

# full_dataset은 Subset이 아닌 ImageFolder 객체여야 target을 가져올 수 있습니다.
# 만약 full_dataset이 ImageFolder 객체였다면:
# train_targets = full_dataset.targets

# 🚨 full_dataset이 random_split 전의 ImageFolder 객체였고, 그 객체가 image_datasets['train']에 해당한다면
# 쌀 모델에서 'train' 데이터셋만 사용했으므로,
# '4. 데이터 로드' 섹션에서 정의된 `full_dataset` 객체를 사용해야 합니다.
# 현재 구조에서는 full_dataset이 ImageFolder 객체이므로 targets 속성이 있습니다.

train_targets = full_dataset.targets # 💡 [수정] full_dataset 객체를 사용해야 합니다.
class_counts = Counter(train_targets)
total_count = sum(class_counts.values())

class_weights = [total_count / class_counts[i] for i in range(NUM_CLASSES)]
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = class_weights / class_weights.min()

print(f"✅ 클래스 가중치 계산 완료: {class_weights.tolist()}")


# ----------------------------------------------------------------------
# 2. EfficientNet-B0 모델 로드 및 분류 레이어 명시적 교체 (수정)
# ----------------------------------------------------------------------

# ImageNet 가중치로 EfficientNet-B0 모델을 로드합니다.
# MODEL_NAME 변수는 이제 'efficientnet-b0'입니다.
model_ft = EfficientNet.from_pretrained(MODEL_NAME).to(device)

# 기존 분류 레이어(_fc)의 입력 피처 수 확인
num_ftrs = model_ft._fc.in_features

# 💡 [수정] 실제 데이터셋의 클래스 수에 맞춰 새로운 Sequential 레이어 정의 및 교체
#      (Dropout(0.5)을 추가하여 과적합 대처)
model_ft._fc = nn.Sequential(
    nn.Dropout(0.5), # 50%의 확률로 Dropout 적용
    nn.Linear(num_ftrs, NUM_CLASSES)
)

model_ft = model_ft.to(device) # 장치에 다시 올립니다.

print(f"✅ EfficientNet-B0의 최종 분류 레이어(w/ Dropout)가 {num_ftrs} -> {NUM_CLASSES}로 성공적으로 교체되었습니다.")


# ----------------------------------------------------------------------
# 3. Loss, Optimizer, Scheduler 정의
# ----------------------------------------------------------------------

# 계산된 class_weights를 GPU/CPU 장치에 올리고 CrossEntropyLoss에 적용
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# AdamW 옵티마이저 사용 (수정된 LEARNING_RATE 적용)
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=LEARNING_RATE)

# 학습률 스케줄러: 8 에포크마다 0.5배 감소로 완화
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.5)

print("✅ Loss, Optimizer, Scheduler 정의 완료.")


✅ 클래스 가중치 계산 완료: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Loaded pretrained weights for efficientnet-b0
✅ EfficientNet-B0의 최종 분류 레이어(w/ Dropout)가 1280 -> 8로 성공적으로 교체되었습니다.
✅ Loss, Optimizer, Scheduler 정의 완료.


In [24]:
# ==============================================================================
# 6. 모델 학습 함수 (수정됨: 'test'를 'val'로 변경)
# ==============================================================================
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f'\nEpoch {epoch+1}/{num_epochs}'); print('-' * 10)

        # 💡 [수정] 'test' 대신 'val' 단계를 반복합니다.
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            # 💡 [수정] 출력 메시지에서 'test'를 'val'로 변경
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # 💡 [수정] 'val' 단계의 정확도가 최고 기록일 때 모델을 저장합니다.
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f'-> New best model found! Saving weights with Acc: {best_acc:.4f}')

        epoch_time = time.time() - epoch_start_time
        print(f'Epoch Time: {epoch_time:.0f}초')

    time_elapsed = time.time() - since
    print(f'\n학습 완료! 총 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초')
    print(f'최고 검증 정확도: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model

In [25]:
# ==============================================================================
# 7. 모델 학습 실행 및 저장
# ==============================================================================

# 모델 학습 시작
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=NUM_EPOCHS)

# Google Drive 마운트
drive.mount('/content/drive')
MODEL_SAVE_PATH = '/content/drive/MyDrive/disease_model.pth' # ⚠️ 파일 이름에 weighted 추가

# 모델 저장
torch.save(model_ft.state_dict(), MODEL_SAVE_PATH)
print(f"\n✅ 최적 모델 가중치가 {MODEL_SAVE_PATH}에 저장되었습니다.")


Epoch 1/10
----------
train Loss: 1.2220 Acc: 0.6191
val Loss: 0.4899 Acc: 0.8156
-> New best model found! Saving weights with Acc: 0.8156
Epoch Time: 300초

Epoch 2/10
----------
train Loss: 0.4608 Acc: 0.8465
val Loss: 0.2974 Acc: 0.8859
-> New best model found! Saving weights with Acc: 0.8859
Epoch Time: 294초

Epoch 3/10
----------
train Loss: 0.3262 Acc: 0.8913
val Loss: 0.2148 Acc: 0.9344
-> New best model found! Saving weights with Acc: 0.9344
Epoch Time: 295초

Epoch 4/10
----------
train Loss: 0.2662 Acc: 0.9115
val Loss: 0.1492 Acc: 0.9547
-> New best model found! Saving weights with Acc: 0.9547
Epoch Time: 289초

Epoch 5/10
----------
train Loss: 0.2320 Acc: 0.9257
val Loss: 0.1584 Acc: 0.9453
Epoch Time: 282초

Epoch 6/10
----------
train Loss: 0.1951 Acc: 0.9352
val Loss: 0.1196 Acc: 0.9500
Epoch Time: 283초

Epoch 7/10
----------
train Loss: 0.1672 Acc: 0.9446
val Loss: 0.1105 Acc: 0.9656
-> New best model found! Saving weights with Acc: 0.9656
Epoch Time: 285초

Epoch 8/10
---

In [27]:
# ==============================================================================
# 8. ONNX 모델 변환 및 저장 (별도 블럭)
# ==============================================================================

!pip install onnx
!pip install onnxruntime

# ⚠️ '7. 모델 학습 실행' 블럭이 실행된 후,
# ⚠️ model_ft, INPUT_SIZE, device 변수가 메모리에 로드된 상태여야 합니다.

ONNX_SAVE_PATH = 'cucumber_disease_model.onnx'

# 1. 모델을 '평가 모드'로 설정 (필수!)
# Dropout, BatchNorm 등이 추론 모드로 변경됩니다.
model_ft.eval()

# 2. 모델에 입력될 더미 텐서(dummy input) 생성
# (EfficientNet-B0 기준 INPUT_SIZE = 224)
# (Batch_size, Channels, Height, Width)
dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)

# 3. ONNX로 내보내기
try:
    torch.onnx.export(
        model_ft,                   # 실행될 모델 (메모리에 로드된 상태)
        dummy_input,                # 모델 입력값 (모양과 타입)
        ONNX_SAVE_PATH,             # 저장될 ONNX 파일 경로
        export_params=True,         # 모델 파라미터(가중치) 저장
        opset_version=11,           # ONNX 오퍼레이터 버전
        do_constant_folding=True,   # 최적화
        input_names = ['input'],    # ONNX 그래프의 입력 텐서 이름
        output_names = ['output'],  # ONNX 그래프의 출력 텐서 이름
        dynamic_axes={'input' : {0 : 'batch_size'},    # 배치 크기를 동적으로 설정
                      'output' : {0 : 'batch_size'}}
    )
    print(f"\n✅ 모델이 ONNX 형식으로 {ONNX_SAVE_PATH}에 성공적으로 저장되었습니다.")

except Exception as e:
    print(f"\n❌ ONNX 변환 중 오류 발생: {e}")

/tmp/ipython-input-2416816508.py:24: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(



✅ 모델이 ONNX 형식으로 cucumber_disease_model.onnx에 성공적으로 저장되었습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')